# 04. Tabular foundation model (TabPFN)

TabPFN is a transformer pretrained on millions of synthetic tables. Instead of being trained on our data, it reads a sample of labelled rows as context and predicts new rows in a single forward pass. This notebook uses TabPFN-3.5-Fast on the same 22 features, splits and metrics as the LightGBM model carried forward (validation PR-AUC 0.3412). TabPFN-3.5 weights are under a non-commercial licence, which allows this evaluation but would require a commercial licence to use the model in a product.

In [1]:
from pathlib import Path

import numpy as np
import torch

from bankguard.data import load_accounts, load_transactions, split_by_date
from bankguard.features import (
    CATEGORICAL,
    FEATURES,
    build_features,
    fit_currency_median,
)

DATA = Path("../data")

tx = load_transactions(DATA / "HI-Small_Trans.csv")
accounts = load_accounts(DATA / "HI-Small_accounts.csv")
splits = split_by_date(tx)
train, val = splits["train"], splits["validation"]

currency_median = fit_currency_median(train)
X = build_features(tx, accounts, currency_median)
X_train, y_train = X.loc[train.index, FEATURES], train["is_laundering"].to_numpy()
X_val, y_val = X.loc[val.index, FEATURES], val["is_laundering"].to_numpy()

del tx, X  # only the two splits are needed from here
print(X_train.shape, y_train.sum(), X_val.shape, y_val.sum())

free, total = torch.cuda.mem_get_info()
print(f"GPU memory free: {free / 1e9:.1f} of {total / 1e9:.1f} GB")


(3248921, 22) 2530 (965524, 22) 1036
GPU memory free: 5.4 of 6.4 GB


The 22 features and splits are the same as for every other model. About 5.4 GB of the laptop's 6 GB of GPU memory is free once the full feature table is deleted.

In [ ]:
import time

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

from bankguard.model import fit_categories, gbm_frame

categories = fit_categories(X_train, CATEGORICAL)
cat_idx = [FEATURES.index(c) for c in CATEGORICAL]


def tabpfn_frame(X):
    """The 22 features as numbers; categories become fixed codes, unseen ones missing."""
    out = gbm_frame(X, FEATURES, categories)
    for c in CATEGORICAL:
        out[c] = out[c].cat.codes.replace(-1, np.nan)
    return out.astype("float32")


# Context: every laundering row plus a random sample of normal rows, 10,000 in total.
rng = np.random.default_rng(42)
pos = np.flatnonzero(y_train == 1)
neg = rng.choice(np.flatnonzero(y_train == 0), size=10_000 - len(pos), replace=False)
ctx = np.sort(np.concatenate([pos, neg]))

clf = TabPFNClassifier.create_default_for_version(ModelVersion.V3_5_FAST)
clf.set_params(device="cuda", categorical_features_indices=cat_idx, random_state=42,
               fit_mode="fit_with_cache")

probe_idx = np.random.default_rng(1).choice(len(X_val), size=20_000, replace=False)

torch.cuda.reset_peak_memory_stats()
start = time.time()
clf.fit(tabpfn_frame(X_train.iloc[ctx]), y_train[ctx])
fit_seconds = time.time() - start

start = time.time()
probe_scores = clf.predict_proba(tabpfn_frame(X_val.iloc[probe_idx]))[:, 1]
predict_seconds = time.time() - start

rows_per_second = len(probe_idx) / predict_seconds
print(f"fit: {fit_seconds:.1f} s | predict 20,000 rows: {predict_seconds:.1f} s")
print(f"estimated time for all validation: {len(X_val) / rows_per_second / 60:.0f} min")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

On the laptop, TabPFN with a 10,000 row context had not finished predicting 20,000 validation rows after 11 minutes, which projects to many hours for all of validation. The cause is that, by default, TabPFN reprocesses the whole context for every chunk of rows it predicts. A second attempt with the context cached (`fit_mode="fit_with_cache"`) was stopped because the laptop was at its limits: GPU memory overflowing into shared system memory, 94% RAM in use and a GPU temperature of 86°C. The run moved to Kaggle (`04b_tabpfn_kaggle.ipynb`).

## Results from Kaggle

In [ ]:
import json
from pathlib import Path

import mlflow
import pandas as pd

tabpfn = json.loads(Path("../models/tabpfn/tabpfn_metrics.json").read_text())
display(pd.DataFrame(tabpfn["probes"]))

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("bankguard-laundering")
with mlflow.start_run(run_name=tabpfn["model"]):
    mlflow.log_params({
        "model_name": "TabPFN-3.5-Fast", "tabpfn_version": "10.2.0", "context_rows": tabpfn["context"],
        "context_laundering_rows": 2530, "fit_mode": "fit_with_cache", "random_state": 42,
        "n_features": 22, "licence": "TabPFN-3.5 licence, non-commercial",
    })
    for key in ["pr_auc", "recall_at_0.1%_fpr", "recall_at_1.0%_fpr"]:
        mlflow.log_metric(key.replace("%", "pct"), tabpfn[key])
    mlflow.log_metric("fit_minutes", tabpfn["fit_minutes"])
    mlflow.log_metric("predict_minutes", tabpfn["predict_minutes"])
    mlflow.log_artifact("../models/tabpfn/tabpfn_metrics.json")
    mlflow.set_tags({"trained_on": "Kaggle, Tesla T4", "notebook": "notebooks/04b_tabpfn_kaggle.ipynb"})

mlflow.search_runs(experiment_names=["bankguard-laundering"])[
    ["tags.mlflow.runName", "metrics.pr_auc", "metrics.recall_at_0.1pct_fpr", "metrics.recall_at_1.0pct_fpr"]
].round(4)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,context,fit_s,predict_20k_s,full_val_min,peak_gb
0,50000,36.5,14.5,12,6.88
1,100000,144.8,27.2,22,8.81
2,200000,681.6,54.0,43,9.21


,tags.mlflow.runName,metrics.pr_auc,metrics.recall_at_0.1pct_fpr,metrics.recall_at_1.0pct_fpr
0,"TabPFN-3.5-Fast, context 200,000",0.3985,0.3967,0.7963
1,"ModernBERT, 4 epochs",0.3526,0.3755,0.7606
2,"ModernBERT, same features as text",0.3023,0.3600,0.7828
3,"DistilBERT, same features as text",0.2281,0.2963,0.7259
4,"LightGBM, owner attributes",0.3412,0.3793,0.7635


With the context cached on a 16 GB GPU, scoring became fast: all of validation in about 5 minutes with a 10,000 row context, and 43 minutes with 200,000. Fit time grows faster than the context (about 3.5 times for each doubling, because attention compares every context row with every other), while peak memory grew from 4.5 GB to 9.2 GB. By the rule fixed in advance, the largest context probed that fitted in memory, 200,000 rows, was used.

TabPFN-3.5-Fast reaches validation PR-AUC 0.398, with recall 0.397 at a 0.1% false positive rate and 0.796 at 1%, the best of all models on every metric (LightGBM: 0.341, 0.379, 0.764). The lead over LightGBM, 0.057 PR-AUC, is about five times LightGBM's spread across seeds. It achieves this without training on our data: it reads 200,000 labelled rows as context and predicts in a single pass.

Two limits apply. The TabPFN-3.5 weights are under a non-commercial licence, so the model can be evaluated but not used in a product without a commercial licence; TabPFN-2, which allows commercial use, is to be tested next. And scoring needs a GPU holding the 200,000 row context, while LightGBM scores on a CPU in milliseconds. Peak memory at 200,000 rows was 9.2 GB of 16, so a larger context might fit; it was not tested.